In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/02/11 15:57:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# !wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-06.csv.gz

In [4]:
# !gzip -dc fhvhv_tripdata_2021-01.csv.gz

In [5]:
# !wc -l fhvhv_tripdata_2021-01.csv

In [ ]:
# df = spark.read \
#     .option("header", "true") \
#     .csv('fhvhv_tripdata_2021-01.csv.gz')

In [7]:
# # this is to show that spark dfs don't infer the types, so they just get type string
# df.schema

In [8]:
# !head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [9]:
# import pandas as pd

In [10]:
# df_pandas = pd.read_csv('head.csv')

In [11]:
# df_pandas.dtypes

In [12]:
# spark.createDataFrame(df_pandas).schema

Integer - 4 bytes
Long - 8 bytes

In [ ]:
from pyspark.sql import types

In [ ]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [15]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [16]:
df = df.repartition(24)

In [ ]:
df.write.parquet('fhvhv/2021/01/')

AnalysisException: path file:/home/filippo/data-engineering-zoomcamp/05-batch/code/fhvhv/2021/01 already exists.

In [18]:
df = spark.read.parquet('fhvhv/2021/01/')

In [19]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [20]:
from pyspark.sql import functions as F

In [21]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [22]:
crazy_stuff('B02884')

's/b44'

In [23]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [24]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-10|  2021-01-10|          97|          25|
|  e/9ce| 2021-01-08|  2021-01-08|         138|         265|
|  e/b3c| 2021-01-01|  2021-01-01|          50|         163|
|  e/9ce| 2021-01-15|  2021-01-15|         163|          79|
|  e/9ce| 2021-01-12|  2021-01-12|          47|          74|
|  s/acd| 2021-01-05|  2021-01-05|          48|          95|
|  e/b38| 2021-01-02|  2021-01-02|          63|          77|
|  e/9ce| 2021-01-06|  2021-01-06|         238|          41|
|  e/acc| 2021-01-02|  2021-01-02|          63|         244|
|  e/acc| 2021-01-24|  2021-01-24|         210|         165|
|  e/b35| 2021-01-16|  2021-01-16|         113|         143|
|  e/9ce| 2021-01-28|  2021-01-28|          91|          89|
|  e/acc| 2021-01-01|  2021-01-01|         231|         148|
|  e/b33| 2021-01-19|  2

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')


DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 34744)
Traceback (most recent call last):
  File "/home/filippo/anaconda3/envs/spark/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/filippo/anaconda3/envs/spark/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/home/filippo/anaconda3/envs/spark/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/filippo/anaconda3/envs/spark/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/home/filippo/spark/spark-3.3.2-bin-hadoop3/python/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/home/filippo/spark/spark-3.3.2-bin-hadoop3/python/pyspark/accumulators.py", line 253, in poll
    if func():
  File

In [50]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag

HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,

HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,

HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,

HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,

HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,

HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,

HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,

HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,

HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,

